# Projet Data Mining — Détection d'intrusions réseau (IDS)

**Dataset :** UNSW-NB15 (adaptable à NSL-KDD, voir cellule CONFIG)
**Cours couvert :** nettoyage → EDA → prétraitement → métriques → validation croisée → classification → clustering → règles d'association

| Étape du projet | Chapitre du cours / TP |
|---|---|
| 1. Chargement et nettoyage | TP1 Raw Data Cleaning, TD1 (valeurs manquantes, doublons) |
| 2. Analyse exploratoire (EDA) | TP2 Pandas / Matplotlib / Seaborn |
| 3. Prétraitement (imputation, encodage, normalisation, split, Pipeline) | TP3, TD1 (Min-Max, Z-score, encodage) |
| 4. Métriques et validation | TP4 (confusion, ROC-AUC, K-Fold, GridSearch, courbes) |
| 5. Classification supervisée | TP5 + cours KNN (KNN from scratch, DT, RF, NB, SVM, LR, SGD, AdaBoost, GB, XGBoost) |
| 6. Malédiction de la dimension, PCA | TD1 (curse of dimensionality) |
| 7. Clustering | TP6 (K-Means, hiérarchique, DBSCAN) |
| 8. Règles d'association | TP7 (Apriori, FP-Growth) |
| 9. Sauvegarde et utilisation du modèle | déploiement |

**Problématique :** à partir des caractéristiques d'un flux réseau, peut-on distinguer un trafic *normal* d'une *attaque* (binaire), puis identifier le type d'attaque (multiclasse) ?

## 0. Setup

In [ ]:
# pip install pandas numpy scikit-learn matplotlib seaborn scipy mlxtend xgboost joblib
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold, ShuffleSplit,
                                     cross_validate, validation_curve, learning_curve, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, classification_report,
                             silhouette_score, adjusted_rand_score)
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from scipy.cluster.hierarchy import linkage, dendrogram

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    HAS_MLXTEND = True
except ImportError:
    HAS_MLXTEND = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("XGBoost:", HAS_XGB, "| mlxtend:", HAS_MLXTEND)

## 1. Configuration et chargement des données

Modifie uniquement cette cellule si ton fichier est différent.
- UNSW-NB15 (version « partitionnée ») : `UNSW_NB15_training-set.csv` et `UNSW_NB15_testing-set.csv`, colonnes `label` (0/1) et `attack_cat`.
- `MAX_ROWS` limite le nombre de lignes (sous-échantillon stratifié) pour que tout s'exécute en un temps raisonnable. Mets `None` pour tout utiliser.

In [ ]:
DATA_FILES  = ["UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"]
LABEL_COL   = "label"        # cible binaire (0 = normal, 1 = attaque)
ATTACK_COL  = "attack_cat"   # type d'attaque (multiclasse), mettre None s'il n'existe pas
ID_COLS     = ["id"]         # colonnes identifiants à supprimer
MAX_ROWS    = 60000          # None = toutes les lignes
SLOW_N      = 12000          # taille d'échantillon pour les méthodes lentes (SVM, clustering, hiérarchique)

dfs = [pd.read_csv(f) for f in DATA_FILES if os.path.exists(f)]
assert dfs, f"Aucun fichier trouvé parmi {DATA_FILES}. Place les CSV dans le dossier du notebook."
df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip().str.lower()
print("Fichiers chargés :", len(dfs), "| Dimensions :", df.shape)
df.head()

## 2. Nettoyage des données (TP1, TD1)

Vérifications : types, valeurs manquantes, valeurs aberrantes de format (`-`), doublons, colonnes inutiles.
**Choix de traitement des valeurs manquantes (TD1) :** imputation par la médiane (numérique, robuste aux outliers fréquents en réseau) et par le mode (catégoriel), faite **dans le Pipeline** pour éviter toute fuite de données (*data leakage*) vers le jeu de test.

In [ ]:
# 2.1 Colonnes inutiles
df = df.drop(columns=[c for c in ID_COLS if c in df.columns])

# 2.2 Cible binaire propre (si la cible est textuelle, ex. NSL-KDD : 'normal' / attaque)
if df[LABEL_COL].dtype == object:
    df[LABEL_COL] = (df[LABEL_COL].str.lower().str.strip() != "normal").astype(int)

# 2.3 Nettoyage du texte des colonnes catégorielles
cat_raw = df.select_dtypes(exclude=np.number).columns
for c in cat_raw:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"-": "none", "nan": np.nan, "": np.nan})   # '-' = 'pas de service' dans UNSW-NB15

# 2.4 Valeurs manquantes
miss = df.isna().sum()
print("Colonnes avec valeurs manquantes :\n", miss[miss > 0] if (miss > 0).any() else "aucune")

# 2.5 Doublons
n_dup = df.duplicated().sum()
print(f"Doublons : {n_dup} ({n_dup/len(df):.1%})")
df = df.drop_duplicates().reset_index(drop=True)

# 2.6 Sous-échantillon stratifié (optionnel)
if MAX_ROWS and len(df) > MAX_ROWS:
    df, _ = train_test_split(df, train_size=MAX_ROWS, stratify=df[LABEL_COL], random_state=RANDOM_STATE)
    df = df.reset_index(drop=True)

print("Dimensions finales :", df.shape)
df.info()

## 3. Analyse exploratoire (EDA) — TP2

### 3.1 Structure et statistiques

In [ ]:
display(df.describe().T.head(25))
display(df.describe(include="object").T)

### 3.2 Déséquilibre des classes

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
df[LABEL_COL].value_counts().plot.bar(ax=ax[0], color=["tab:green", "tab:red"])
ax[0].set_title("Normal (0) vs Attaque (1)"); ax[0].set_xticklabels(["Attaque" if i == 1 else "Normal" for i in df[LABEL_COL].value_counts().index], rotation=0)
df[LABEL_COL].value_counts().plot.pie(ax=ax[1], autopct="%.1f%%", labels=["Attaque" if i == 1 else "Normal" for i in df[LABEL_COL].value_counts().index])
ax[1].set_ylabel("")
if ATTACK_COL and ATTACK_COL in df.columns:
    df[ATTACK_COL].value_counts().plot.barh(ax=ax[2]); ax[2].set_title("Types d'attaques")
plt.tight_layout(); plt.show()
print(df[LABEL_COL].value_counts(normalize=True).round(3))

### 3.3 Variables numériques : distributions, asymétrie, outliers

In [ ]:
num_all = df.select_dtypes(include=np.number).columns.drop(LABEL_COL).tolist()
skew = df[num_all].skew().sort_values(ascending=False)
print("Variables les plus asymétriques :\n", skew.head(10).round(1))

top = [c for c in ["dur", "sbytes", "dbytes", "rate", "spkts", "sttl"] if c in num_all] or num_all[:6]
fig, axes = plt.subplots(2, len(top), figsize=(4 * len(top), 7))
for i, c in enumerate(top):
    sns.histplot(df[c], bins=40, ax=axes[0, i]); axes[0, i].set_title(c)
    sns.histplot(np.log1p(df[c].clip(lower=0)), bins=40, ax=axes[1, i], color="tab:orange"); axes[1, i].set_title(f"log1p({c})")
plt.tight_layout(); plt.show()

# Outliers (règle IQR)
q1, q3 = df[num_all].quantile(0.25), df[num_all].quantile(0.75)
iqr = q3 - q1
out_rate = ((df[num_all] < q1 - 1.5 * iqr) | (df[num_all] > q3 + 1.5 * iqr)).mean().sort_values(ascending=False)
print("\nTaux d'outliers (IQR), top 8 :\n", out_rate.head(8).round(3))

**Lecture :** en trafic réseau, les distributions sont très asymétriques (queues lourdes) et les « outliers » sont souvent des attaques réelles : on ne les supprime donc **pas**. On applique plutôt une transformation `log1p` avant la normalisation.

### 3.4 Variables vs cible

In [ ]:
fig, axes = plt.subplots(1, len(top[:3]), figsize=(5 * len(top[:3]), 4))
for ax, c in zip(np.atleast_1d(axes), top[:3]):
    sns.boxplot(x=df[LABEL_COL], y=np.log1p(df[c].clip(lower=0)), ax=ax); ax.set_title(f"log1p({c}) par classe")
plt.tight_layout(); plt.show()

cat_cols_eda = df.select_dtypes(exclude=np.number).columns.drop(ATTACK_COL, errors="ignore").tolist()
fig, axes = plt.subplots(1, len(cat_cols_eda), figsize=(6 * len(cat_cols_eda), 4))
for ax, c in zip(np.atleast_1d(axes), cat_cols_eda):
    top_vals = df[c].value_counts().head(8).index
    (df[df[c].isin(top_vals)].groupby(c)[LABEL_COL].mean().sort_values().plot.barh(ax=ax))
    ax.set_title(f"Taux d'attaque par {c} (top 8)"); ax.set_xlabel("proportion d'attaques")
plt.tight_layout(); plt.show()

### 3.5 Corrélations

In [ ]:
corr = df[num_all + [LABEL_COL]].corr()
cor_target = corr[LABEL_COL].drop(LABEL_COL).sort_values(key=abs, ascending=False)
print("Top 10 corrélations avec la cible :\n", cor_target.head(10).round(3))

sel = cor_target.head(15).index.tolist() + [LABEL_COL]
plt.figure(figsize=(10, 8))
sns.heatmap(df[sel].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Heatmap des 15 variables les plus corrélées à la cible"); plt.show()

# Paires de variables fortement redondantes (|r| > 0.95)
c_abs = corr.drop(index=LABEL_COL, columns=LABEL_COL).abs()
pairs = [(a, b, c_abs.loc[a, b]) for i, a in enumerate(c_abs.columns) for b in c_abs.columns[i+1:] if c_abs.loc[a, b] > 0.95]
print(f"\n{len(pairs)} paires redondantes (|r|>0.95), exemples :", [(a, b, round(r, 2)) for a, b, r in pairs[:5]])

## 4. Prétraitement (TP3, TD1)

- **Séparation train/test stratifiée** avant tout calcul de statistiques (anti-fuite).
- **Numériques :** imputation médiane → `log1p` (si asymétrie > 2 et valeurs ≥ 0) → **Z-score** (`StandardScaler`). Le scaling est indispensable pour KNN, SVM, K-Means (TD1 : distance dominée par l'attribut de plus grande échelle).
- **Catégorielles :** imputation mode → **One-Hot** (`proto`, `service`, `state` sont **nominales**, sans ordre : le label encoding introduirait un ordre artificiel). Les modalités rares (<1 %) sont regroupées pour limiter la dimension.
- Le tout dans un `ColumnTransformer` + `Pipeline` (reproductible, sans leakage, réutilisable en production).

In [ ]:
y = df[LABEL_COL].astype(int)
y_attack = df[ATTACK_COL] if (ATTACK_COL and ATTACK_COL in df.columns) else None
X = df.drop(columns=[c for c in [LABEL_COL, ATTACK_COL] if c and c in df.columns])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Taux d'attaque train / test :", y_train.mean().round(3), y_test.mean().round(3))

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
sk = X_train[num_cols].skew()
log_cols = [c for c in num_cols if sk[c] > 2 and X_train[c].min() >= 0]
lin_cols = [c for c in num_cols if c not in log_cols]
print(f"{len(log_cols)} variables log-transformées | {len(lin_cols)} linéaires | {len(cat_cols)} catégorielles: {cat_cols}")

def make_preprocessor():
    num_log = Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                        ("sc", StandardScaler())])
    num_lin = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01, sparse_output=False))])
    return ColumnTransformer([("num_log", num_log, log_cols), ("num_lin", num_lin, lin_cols), ("cat", cat, cat_cols)])

prep = make_preprocessor().fit(X_train)
Xtr_p, Xte_p = prep.transform(X_train), prep.transform(X_test)
feat_names = prep.get_feature_names_out()
print("Dimension après prétraitement :", Xtr_p.shape)

### 4.1 Rappel TD1 : Min-Max vs Z-score sur un exemple

In [ ]:
d = np.array([200, 300, 400, 600, 1000])
print("Min-Max :", ((d - d.min()) / (d.max() - d.min())).round(3))
print("Z-score :", ((d - d.mean()) / d.std()).round(3))

## 5. Métriques d'évaluation (TP4)

Dans un IDS, l'**accuracy seule est insuffisante** (classes déséquilibrées, coûts asymétriques) :
- **Recall (taux de détection)** = TP/(TP+FN) : proportion d'attaques détectées. Un FN = intrusion manquée (coûteux).
- **Précision** = TP/(TP+FP) : fiabilité des alertes.
- **FAR (taux de fausses alarmes)** = FP/(FP+TN) : trafic normal bloqué à tort.
- **F1** : compromis précision/recall. **ROC-AUC** : qualité du score indépendamment du seuil.

In [ ]:
results, fitted, scores = [], {}, {}

def evaluate(name, pipe, Xtr, ytr, Xte, yte):
    t0 = time.time(); pipe.fit(Xtr, ytr); fit_t = time.time() - t0
    t0 = time.time(); y_pred = pipe.predict(Xte); pred_t = time.time() - t0
    if hasattr(pipe, "predict_proba"):      s = pipe.predict_proba(Xte)[:, 1]
    elif hasattr(pipe, "decision_function"): s = pipe.decision_function(Xte)
    else:                                    s = y_pred
    tn, fp, fn, tp = confusion_matrix(yte, y_pred).ravel()
    row = dict(Modèle=name, Accuracy=accuracy_score(yte, y_pred), Précision=precision_score(yte, y_pred),
               Recall=recall_score(yte, y_pred), F1=f1_score(yte, y_pred), AUC=roc_auc_score(yte, s),
               FAR=fp / (fp + tn), FN=fn, **{"Fit (s)": fit_t, "Predict (s)": pred_t})
    results.append(row); fitted[name] = pipe; scores[name] = s
    return row

## 6. Classification supervisée (TP5 + cours KNN)

### 6.1 KNN — implémentation « from scratch » (pseudo-code du cours) puis scikit-learn
Étapes du cours : (1) distance de x à tous les points d'entraînement, (2) tri croissant, (3) sélection des k plus proches, (4) comptage des classes, (5) vote majoritaire.

In [ ]:
def knn_scratch(X_tr, y_tr, X_new, k=5, p=2):
    X_tr, y_tr = np.asarray(X_tr), np.asarray(y_tr)
    preds = []
    for x in X_new:
        dist = (np.abs(X_tr - x) ** p).sum(axis=1) ** (1 / p)     # Minkowski (p=1 Manhattan, p=2 Euclidienne)
        idx = np.argsort(dist)[:k]                                  # k plus proches
        votes = np.bincount(y_tr[idx])                              # vote majoritaire
        preds.append(votes.argmax())
    return np.array(preds)

n_check = 500
t0 = time.time()
p_scratch = knn_scratch(Xtr_p[:SLOW_N], y_train.values[:SLOW_N], Xte_p[:n_check], k=5)
print(f"KNN scratch : {time.time()-t0:.1f}s | accuracy sur {n_check} points : {accuracy_score(y_test.values[:n_check], p_scratch):.3f}")
sk_knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr_p[:SLOW_N], y_train.values[:SLOW_N])
print("Accord avec scikit-learn :", (sk_knn.predict(Xte_p[:n_check]) == p_scratch).mean())

# Distances du TD1 : Minkowski p = 1, 2, inf
a, b = np.array([2, 5]), np.array([5, 1])
print("\nTD1 -> Manhattan:", np.abs(a-b).sum(), "| Euclidienne:", np.sqrt(((a-b)**2).sum()).round(3), "| Chebyshev:", np.abs(a-b).max())

### 6.2 Entraînement de tous les modèles du cours (même Pipeline, même split)

In [ ]:
def make_models():
    m = {
        "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "Decision Tree":       DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE),
        "Random Forest":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
        "Naive Bayes":         GaussianNB(),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "SGD Classifier":      SGDClassifier(loss="log_loss", random_state=RANDOM_STATE),
        "SVM (RBF)":           SVC(kernel="rbf"),
        "AdaBoost":            AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE),
        "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }
    if HAS_XGB:
        m["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric="logloss",
                                     n_jobs=-1, random_state=RANDOM_STATE)
    return m

for name, model in make_models().items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", model)])
    if name == "SVM (RBF)":   # SVM : coût O(n^2) -> sous-échantillon d'entraînement
        idx = np.random.RandomState(RANDOM_STATE).choice(len(X_train), min(SLOW_N, len(X_train)), replace=False)
        row = evaluate(name, pipe, X_train.iloc[idx], y_train.iloc[idx], X_test, y_test)
    else:
        row = evaluate(name, pipe, X_train, y_train, X_test, y_test)
    print(f"{name:20s} F1={row['F1']:.4f}  Recall={row['Recall']:.4f}  AUC={row['AUC']:.4f}  ({row['Fit (s)']:.1f}s)")

### 6.3 Comparaison des modèles (TP5, section « Which algorithm to choose ? »)

In [ ]:
res = pd.DataFrame(results).set_index("Modèle").sort_values("F1", ascending=False)
display(res.round(4))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
res[["Accuracy", "Précision", "Recall", "F1", "AUC"]].plot.bar(ax=ax[0]); ax[0].set_title("Métriques par modèle"); ax[0].set_ylim(res[["Accuracy","F1"]].min().min() - 0.05, 1.0)
ax[0].tick_params(axis="x", rotation=45)
for n, s in scores.items():
    fpr, tpr, _ = roc_curve(y_test, s); ax[1].plot(fpr, tpr, label=f"{n} ({roc_auc_score(y_test, s):.3f})")
ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].set_title("Courbes ROC"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

best_name = res.index[0]
print("Meilleur modèle (F1) :", best_name)
ConfusionMatrixDisplay.from_estimator(fitted[best_name], X_test, y_test, display_labels=["Normal", "Attaque"], cmap="Blues")
plt.title(f"Matrice de confusion — {best_name}"); plt.show()

### 6.4 Arbre de décision interprétable et importance des variables

In [ ]:
dt_pipe = Pipeline([("prep", make_preprocessor()), ("clf", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE))]).fit(X_train, y_train)
plt.figure(figsize=(20, 8))
plot_tree(dt_pipe["clf"], feature_names=dt_pipe["prep"].get_feature_names_out(), class_names=["Normal", "Attaque"], filled=True, fontsize=8)
plt.show()

rf = fitted["Random Forest"]
imp = pd.Series(rf["clf"].feature_importances_, index=rf["prep"].get_feature_names_out()).sort_values(ascending=False).head(15)
imp[::-1].plot.barh(figsize=(8, 6), title="Random Forest : top 15 variables importantes"); plt.show()

## 7. Validation croisée, courbes et optimisation (TP4)

### 7.1 K-Fold, Stratified K-Fold, ShuffleSplit
*Leave-One-Out* est décrit dans le cours mais **impraticable ici** (n entraînements, une par ligne) : on le justifie plutôt que de l'exécuter.

In [ ]:
idx_cv = np.random.RandomState(RANDOM_STATE).choice(len(X_train), min(SLOW_N * 2, len(X_train)), replace=False)
Xcv, ycv = X_train.iloc[idx_cv], y_train.iloc[idx_cv]
pipe_dt = Pipeline([("prep", make_preprocessor()), ("clf", DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE))])

splitters = {"KFold(5)": KFold(5, shuffle=True, random_state=RANDOM_STATE),
             "StratifiedKFold(5)": StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
             "ShuffleSplit(5)": ShuffleSplit(5, test_size=0.2, random_state=RANDOM_STATE)}
rows = []
for n, cv in splitters.items():
    r = cross_validate(pipe_dt, Xcv, ycv, cv=cv, scoring=["accuracy", "f1", "recall"])
    rows.append({"CV": n, "F1 moyen": r["test_f1"].mean(), "F1 écart-type": r["test_f1"].std(), "Recall moyen": r["test_recall"].mean()})
display(pd.DataFrame(rows).round(4))

### 7.2 Validation croisée stratifiée (5 plis) de tous les modèles rapides

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = {}
for name, model in make_models().items():
    if name in ("SVM (RBF)", "Gradient Boosting", "AdaBoost"):   # lents : exclus de la CV complète
        continue
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", model)])
    cv_scores[name] = cross_validate(pipe, Xcv, ycv, cv=skf, scoring="f1")["test_score"]
cvdf = pd.DataFrame(cv_scores)
display(cvdf.agg(["mean", "std"]).T.round(4))
plt.figure(figsize=(10, 4)); sns.boxplot(data=cvdf); plt.xticks(rotation=30); plt.title("F1 par pli (Stratified 5-Fold)"); plt.show()

### 7.3 Courbes de validation (sur/sous-apprentissage)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
depths = [1, 2, 3, 5, 8, 12, 16, 20]
tr, va = validation_curve(pipe_dt, Xcv, ycv, param_name="clf__max_depth", param_range=depths, cv=3, scoring="f1")
ax[0].plot(depths, tr.mean(1), "o-", label="train"); ax[0].plot(depths, va.mean(1), "o-", label="validation")
ax[0].set_xlabel("max_depth"); ax[0].set_ylabel("F1"); ax[0].set_title("Decision Tree : profondeur"); ax[0].legend()

pipe_knn = Pipeline([("prep", make_preprocessor()), ("clf", KNeighborsClassifier())])
ks = [1, 3, 5, 7, 11, 15, 21]
tr, va = validation_curve(pipe_knn, Xcv.iloc[:SLOW_N], ycv.iloc[:SLOW_N], param_name="clf__n_neighbors", param_range=ks, cv=3, scoring="f1")
ax[1].plot(ks, tr.mean(1), "o-", label="train"); ax[1].plot(ks, va.mean(1), "o-", label="validation")
ax[1].set_xlabel("k"); ax[1].set_title("KNN : choix de k (petit k = surapprentissage)"); ax[1].legend()
plt.tight_layout(); plt.show()

### 7.4 GridSearchCV (optimisation des hyperparamètres)

In [ ]:
pipe_rf = Pipeline([("prep", make_preprocessor()), ("clf", RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))])
grid = {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 15], "clf__min_samples_leaf": [1, 3]}
gs = GridSearchCV(pipe_rf, grid, cv=3, scoring="f1", n_jobs=-1, refit=True).fit(Xcv, ycv)
print("Meilleurs paramètres :", gs.best_params_, "| F1 CV :", round(gs.best_score_, 4))
display(pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head(8))

# Re-entraînement du modèle optimal sur tout le train puis évaluation finale sur le test
tuned = Pipeline([("prep", make_preprocessor()), ("clf", RandomForestClassifier(
    n_estimators=gs.best_params_["clf__n_estimators"], max_depth=gs.best_params_["clf__max_depth"],
    min_samples_leaf=gs.best_params_["clf__min_samples_leaf"], n_jobs=-1, random_state=RANDOM_STATE))])
evaluate("Random Forest (tuned)", tuned, X_train, y_train, X_test, y_test)
pd.DataFrame(results).set_index("Modèle").sort_values("F1", ascending=False).round(4).head(5)

### 7.5 Courbe d'apprentissage (a-t-on assez de données ?)

In [ ]:
sizes, tr, va = learning_curve(tuned, Xcv, ycv, train_sizes=[0.1, 0.3, 0.5, 0.7, 1.0], cv=3, scoring="f1", n_jobs=-1)
plt.figure(figsize=(7, 4.5))
plt.plot(sizes, tr.mean(1), "o-", label="train"); plt.plot(sizes, va.mean(1), "o-", label="validation")
plt.xlabel("Taille du train"); plt.ylabel("F1"); plt.title("Learning curve — Random Forest"); plt.legend(); plt.show()

## 8. Classification multiclasse : identifier le type d'attaque

In [ ]:
if y_attack is not None:
    ya_train, ya_test = y_attack.loc[X_train.index], y_attack.loc[X_test.index]
    multi = Pipeline([("prep", make_preprocessor()), ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=RANDOM_STATE))]).fit(X_train, ya_train)
    yp = multi.predict(X_test)
    print(classification_report(ya_test, yp, zero_division=0))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay.from_predictions(ya_test, yp, normalize="true", xticks_rotation=45, ax=ax, cmap="Blues", values_format=".2f")
    plt.title("Matrice de confusion normalisée (types d'attaques)"); plt.tight_layout(); plt.show()
else:
    print("Pas de colonne de type d'attaque : section ignorée.")

## 9. Malédiction de la dimension, PCA et sélection de variables (TD1)

La distance euclidienne perd son pouvoir discriminant quand le nombre de dimensions augmente (les distances se concentrent) → KNN, K-Means et DBSCAN se dégradent. Remèdes : **PCA** (réduction) et **sélection de variables**.

In [ ]:
# Démonstration : concentration des distances
rng = np.random.RandomState(0)
dims = [2, 5, 10, 50, 100, 500, 1000]
ratio = []
for d in dims:
    P = rng.rand(500, d); q = rng.rand(1, d)
    dist = np.linalg.norm(P - q, axis=1); ratio.append((dist.max() - dist.min()) / dist.min())
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ax[0].plot(dims, ratio, "o-"); ax[0].set_xscale("log"); ax[0].set_title("(dmax-dmin)/dmin vs dimension"); ax[0].set_xlabel("dimensions")

# PCA : variance expliquée
pca = PCA().fit(Xtr_p)
cum = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.argmax(cum >= 0.95) + 1)
ax[1].plot(cum); ax[1].axhline(0.95, color="r", ls="--"); ax[1].set_title(f"PCA : {n95} composantes pour 95 % de variance"); ax[1].set_xlabel("composantes")

# KNN sur espace réduit
accs = {}
for n in [2, 5, 10, n95, Xtr_p.shape[1]]:
    n = min(n, Xtr_p.shape[1])
    if n < Xtr_p.shape[1]:
        pc = PCA(n_components=n, random_state=RANDOM_STATE).fit(Xtr_p[:SLOW_N]); A, B = pc.transform(Xtr_p[:SLOW_N]), pc.transform(Xte_p)
    else:
        A, B = Xtr_p[:SLOW_N], Xte_p
    accs[n] = f1_score(y_test, KNeighborsClassifier(5, n_jobs=-1).fit(A, y_train.values[:SLOW_N]).predict(B))
ax[2].plot(list(accs.keys()), list(accs.values()), "o-"); ax[2].set_title("F1 KNN selon le nombre de composantes PCA"); ax[2].set_xlabel("composantes")
plt.tight_layout(); plt.show()

# Information mutuelle : sélection de variables
mi = pd.Series(mutual_info_classif(Xtr_p[:SLOW_N], y_train.values[:SLOW_N], random_state=RANDOM_STATE), index=feat_names).sort_values(ascending=False)
mi.head(15)[::-1].plot.barh(figsize=(8, 5), title="Top 15 variables (information mutuelle)"); plt.show()

## 10. Clustering non supervisé (TP6)

Objectif IDS : les attaques forment-elles des groupes naturels *sans* utiliser l'étiquette ? On compare ensuite les clusters aux vraies classes (évaluation externe : ARI, pureté) en plus de l'évaluation interne (silhouette).

In [ ]:
rs = np.random.RandomState(RANDOM_STATE)
ci = rs.choice(len(Xtr_p), min(SLOW_N, len(Xtr_p)), replace=False)
Xc, yc = Xtr_p[ci], y_train.values[ci]
Xc2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xc)   # pour la visualisation

# 10.1 K-Means : coude + silhouette
ks = range(2, 11); inertia, sil = [], []
sub = rs.choice(len(Xc), min(3000, len(Xc)), replace=False)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xc)
    inertia.append(km.inertia_); sil.append(silhouette_score(Xc[sub], km.labels_[sub]))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertia, "o-"); ax[0].set_title("Méthode du coude (inertie)"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), sil, "o-"); ax[1].set_title("Score de silhouette"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()
best_k = list(ks)[int(np.argmax(sil))]
print("k optimal (silhouette) :", best_k)

In [ ]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit(Xc)
print("ARI vs classes réelles :", round(adjusted_rand_score(yc, km.labels_), 3))
ct = pd.crosstab(km.labels_, yc, rownames=["cluster"], colnames=["classe"]); display(ct)
print("Pureté :", round(ct.max(axis=1).sum() / ct.values.sum(), 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=Xc2[:, 0], y=Xc2[:, 1], hue=km.labels_, palette="tab10", s=8, ax=ax[0]); ax[0].set_title(f"K-Means (k={best_k}) — projection PCA")
sns.scatterplot(x=Xc2[:, 0], y=Xc2[:, 1], hue=np.where(yc == 1, "Attaque", "Normal"), palette={"Normal": "tab:green", "Attaque": "tab:red"}, s=8, ax=ax[1]); ax[1].set_title("Classes réelles")
plt.tight_layout(); plt.show()

### 10.2 Clustering hiérarchique

In [ ]:
h = rs.choice(len(Xc), min(300, len(Xc)), replace=False)
Z = linkage(Xc[h], method="ward")
plt.figure(figsize=(12, 4)); dendrogram(Z, truncate_mode="lastp", p=25, leaf_rotation=90); plt.title("Dendrogramme (Ward, 300 points)"); plt.show()

h2 = rs.choice(len(Xc), min(3000, len(Xc)), replace=False)   # AgglomerativeClustering est O(n^2) en mémoire
for link in ["ward", "complete", "average"]:
    lab = AgglomerativeClustering(n_clusters=best_k, linkage=link).fit_predict(Xc[h2])
    print(f"{link:9s} silhouette={silhouette_score(Xc[h2], lab):.3f}  ARI={adjusted_rand_score(yc[h2], lab):.3f}")

### 10.3 DBSCAN : les points « bruit » comme anomalies
DBSCAN n'exige pas k et isole le bruit (points de faible densité) : c'est une approche naturelle de **détection d'anomalies**.

In [ ]:
Xd = PCA(n_components=min(10, Xc.shape[1]), random_state=RANDOM_STATE).fit_transform(Xc)
min_samples = 2 * Xd.shape[1]
kd = np.sort(NearestNeighbors(n_neighbors=min_samples).fit(Xd).kneighbors(Xd)[0][:, -1])
eps = float(np.percentile(kd, 90))
plt.figure(figsize=(6, 4)); plt.plot(kd); plt.axhline(eps, color="r", ls="--", label=f"eps ≈ {eps:.2f}")
plt.title("k-distance graph"); plt.legend(); plt.show()

db = DBSCAN(eps=eps, min_samples=min_samples).fit(Xd)
lab = db.labels_; noise = lab == -1
print(f"Clusters : {len(set(lab)) - (1 if noise.any() else 0)} | Bruit : {noise.mean():.1%}")
if noise.any():
    print(f"Taux d'attaque parmi le bruit : {yc[noise].mean():.1%}  vs  dans les clusters : {yc[~noise].mean():.1%}  (global : {yc.mean():.1%})")
X2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xd)
sns.scatterplot(x=X2[:, 0], y=X2[:, 1], hue=np.where(noise, "bruit", lab.astype(str)), s=8)
plt.title("DBSCAN (projection PCA)"); plt.show()

## 11. Règles d'association (TP7) — Apriori et FP-Growth

On transforme les flux en « transactions » : les variables numériques sont **discrétisées** (faible/moyen/élevé par terciles), puis tout est one-hot encodé. On cherche des règles du type `{proto=tcp, sbytes=élevé} → {label=attack}` et on les classe par **lift** (>1 : association positive).

In [ ]:
if HAS_MLXTEND:
    tx = df.sample(min(20000, len(df)), random_state=RANDOM_STATE).copy()
    bins_cols = [c for c in ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sttl", "ct_srv_src"] if c in tx.columns]
    items = pd.DataFrame(index=tx.index)
    for c in bins_cols:
        items[c] = pd.qcut(tx[c].rank(method="first"), q=3, labels=["faible", "moyen", "élevé"]).astype(str)
    for c in [c for c in ["proto", "service", "state"] if c in tx.columns]:
        top_vals = tx[c].value_counts().head(6).index
        items[c] = tx[c].where(tx[c].isin(top_vals), "autre")
    items["label"] = np.where(tx[LABEL_COL] == 1, "attack", "normal")
    basket = pd.get_dummies(items, prefix_sep="=").astype(bool)
    print("Matrice transactions :", basket.shape)

    t0 = time.time(); fi_ap = apriori(basket, min_support=0.1, use_colnames=True, max_len=4); t_ap = time.time() - t0
    t0 = time.time(); fi_fp = fpgrowth(basket, min_support=0.1, use_colnames=True, max_len=4); t_fp = time.time() - t0
    print(f"Apriori : {len(fi_ap)} itemsets en {t_ap:.2f}s | FP-Growth : {len(fi_fp)} itemsets en {t_fp:.2f}s")

    def get_rules(freq, **kw):
        try:    return association_rules(freq, num_itemsets=len(basket), **kw)   # mlxtend >= 0.23.2
        except TypeError: return association_rules(freq, **kw)

    rules = get_rules(fi_fp, metric="confidence", min_threshold=0.8)
    to_attack = rules[rules["consequents"].apply(lambda s: s == frozenset({"label=attack"}))]
    to_normal = rules[rules["consequents"].apply(lambda s: s == frozenset({"label=normal"}))]
    show = lambda r: r.assign(antecedents=r.antecedents.apply(lambda s: ", ".join(sorted(s))))[["antecedents", "support", "confidence", "lift"]]
    print(f"{len(rules)} règles (conf >= 0.8)")
    print("\nRègles → ATTAQUE (top lift) :"); display(show(to_attack.sort_values(["lift", "confidence"], ascending=False).head(10)).round(3))
    print("Règles → NORMAL (top lift) :");  display(show(to_normal.sort_values(["lift", "confidence"], ascending=False).head(10)).round(3))
else:
    print("mlxtend manquant : exécute `pip install mlxtend` puis relance cette cellule.")

## 12. Sauvegarde et utilisation du modèle (déploiement)

In [ ]:
import joblib
res = pd.DataFrame(results).set_index("Modèle").sort_values("F1", ascending=False)
final_name = res.index[0]
final_model = fitted[final_name]
joblib.dump(final_model, "ids_best_model.joblib")
print("Modèle sauvegardé :", final_name)

def detect(flows: pd.DataFrame):
    """Prend des flux réseau bruts (mêmes colonnes que X) et renvoie la prédiction + score d'alerte."""
    m = joblib.load("ids_best_model.joblib")
    pred = m.predict(flows)
    score = m.predict_proba(flows)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(flows)
    return pd.DataFrame({"prediction": np.where(pred == 1, "ATTAQUE", "normal"), "score": np.round(score, 3)}, index=flows.index)

detect(X_test.head(8)).join(y_test.head(8).rename("réel"))

## 13. Conclusion (à adapter avec tes chiffres)

1. **Données :** déséquilibre et asymétrie fortes → `log1p` + Z-score, One-Hot avec regroupement des modalités rares, outliers conservés car souvent des attaques.
2. **Classification :** les méthodes d'ensemble (Random Forest, Gradient Boosting, XGBoost) dominent en F1/AUC ; KNN et SVM sont sensibles au scaling et coûteux en prédiction ; Naive Bayes est rapide mais limité par l'hypothèse d'indépendance. Comparer **recall et FAR**, pas seulement l'accuracy.
3. **Validation :** Stratified K-Fold, courbes de validation (profondeur, k) et GridSearchCV limitent le surapprentissage ; la learning curve indique si plus de données aideraient.
4. **Dimension :** la PCA réduit fortement la dimension avec peu de perte, ce qui illustre la malédiction de la dimension pour KNN/K-Means.
5. **Clustering :** K-Means/hiérarchique séparent partiellement normal et attaque (voir ARI et pureté) ; DBSCAN isole des anomalies (bruit) dont le taux d'attaque est supérieur à la moyenne.
6. **Règles d'association :** des règles à fort lift relient protocole/état/volume au caractère malveillant, utiles pour écrire des signatures.
7. **Limites :** décalage de distribution train/test dans UNSW-NB15, jeu de données synthétique (laboratoire) donc généralisation à un réseau réel non garantie ; extensions possibles : SMOTE, réseaux de neurones, détection de nouvelles attaques (*zero-day*).